# StatCore PCA: reliability-oriented principal component analysis

[Open in Colab](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/statcore-pca.ipynb)

**Paper topic:** Reliability-Oriented Principal Component Analysis in StatCore — `docs/latex/statcore-pca.tex` ([repository](https://github.com/Jabualani/ML-Reliabilty/blob/main/docs/latex/statcore-pca.tex)).
**Experiment script reproduced here:** `docs/latex/experiments/pca_experiment.py` (reduced seed count so the notebook runs in a few minutes).

StatCore is a pure-NumPy/SciPy model family in `ml_reliability.statcore`. Every model has a `variant`
argument; each variant below is compared with the standard reference implementation.
**scikit-learn is used only as the baseline/reference** (and for loading public datasets), never inside StatCore.

## 1. Install (parameter cell)

> **Note:** StatCore has **not been pushed / published yet**. Until the repository is public, the
> `git+https` install below will fail. In that case use the **fallback cell**: zip the repository folder,
> upload the zip to Colab (left sidebar → Files, or the `files.upload()` helper) and `pip install` it.
> Run exactly one of the two cells.

In [ ]:
# @title Install StatCore (PARAMETER: change the source if needed)
# Option A - once the repository is published:
!pip install "git+https://github.com/Jabualani/ML-Reliabilty.git"

In [ ]:
# @title FALLBACK install from an uploaded zip of the repository (use if Option A fails)
# Zip the repo folder (ML-Reliabilty.zip), then uncomment and run:
# from google.colab import files
# files.upload()                       # pick ML-Reliabilty.zip
# !unzip -q -o ML-Reliabilty.zip -d repo
# !pip install ./repo/ML-Reliabilty     # adjust to the folder that contains pyproject.toml

In [ ]:
# @title Parameters and imports
SEEDS = (11, 23, 37)   # the paper uses more seeds; fewer here to keep the run short

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## 2. Method summary

Notation: $X\in\mathbb{R}^{n\times d}$, $k$ components, centred data $X_c = X-\mu$, weights $w_i\ge 0$.
`transform` returns $Z=(x-\mu)\,V_k^{+}$ and `inverse_transform` returns $\hat x = z\,V_k+\mu$, where $V_k\in\mathbb{R}^{k\times d}$ holds the components as rows (`components_`).

**Reference / `base`.** Thin SVD $X_c=U\Sigma V^\top$, $V_k$ = first $k$ right singular vectors, $\mu=\bar x$. This minimises the *training* squared reconstruction error $\|X_c-X_cV_k^\top V_k\|_F^2$.

**`supervised_row_aware`** (weighted PCA). With scores $s_i=V_k(x_i-\bar x)$ from a plain SVD and $\lambda_j=\sigma_j^2/n$:
$$d_i^2=\sum_{j\le k}\frac{s_{ij}^2}{\lambda_j},\qquad c=2.5\,\mathrm{median}(d^2),\qquad w_i=\begin{cases}1 & d_i^2\le c\\ c/d_i^2 & d_i^2>c\end{cases}$$
If labels are given, $w_i\leftarrow w_i\,\dfrac{n}{C\,n_{y_i}}$ (inverse class frequency). Then $w\leftarrow nw/\sum_i w_i$, the mean is $\mu_w=\sum_i w_ix_i/\sum_i w_i$ and components are the right singular vectors of $\mathrm{diag}(\sqrt w)(X-\mu_w)$.

**`robust`** (median start, Huber-type IRLS on orthogonal residuals). $\mu_0=\mathrm{median}(X)$; initial scaled norms $r_i=\|(x_i-\mu_0)/(1.4826\,\mathrm{MAD}_j)\|$, $c_0=\mathrm{med}(r)+1.345\cdot1.4826\,\mathrm{MAD}(r)$, $w_i=\min(1,c_0/r_i)^2$. For at most 10 iterations: weighted mean $\mu_w$, weighted SVD, residual norms $r_i=\|(x_i-\mu_w)-(x_i-\mu_w)V_k^\top V_k\|$, $c=\mathrm{med}(r)+1.345\cdot1.4826\,\mathrm{MAD}(r)$, $w_i=\min(1,c/r_i)$. *Implementation detail:* the stored `mean_` used by `transform` is the coordinate-wise **median** $\mu_0$.

**`sparse`** (L1 power iterations with deflation). For each component, start from the leading singular vector of the residual matrix $R$ (plus tiny noise) and iterate
$u=Rv/\|Rv\|,\; z=R^\top u,\; v\leftarrow \mathrm{soft}(z,\alpha\,\|z\|_\infty)/\|\cdot\|$, then deflate $R\leftarrow R-(Rv)v^\top$. Components are sparse but **not orthogonal**, so `transform` uses the pseudo-inverse of the component matrix.

**`probabilistic`** (closed-form PPCA, Tipping & Bishop). With eigenvalues $\ell_1\ge\dots\ge\ell_d$ of the sample covariance:
$$\sigma^2=\frac{1}{d-k}\sum_{j>k}\ell_j,\qquad W=V_k^\top(\Lambda_k-\sigma^2I)^{1/2},\qquad M=W^\top W+\sigma^2I,\qquad z=M^{-1}W^\top(x-\mu).$$
Reconstruction is the orthogonal projection onto $\mathrm{span}(W)$: $\hat x=\mu+zMW^{+}$.

Because the reference PCA is the *exact minimiser of training reconstruction error*, no variant can beat it in-sample on clean data; variants trade a little clean accuracy for robustness to contamination, class balance, sparsity or a noise model.

## 3. Experiment (mirrors `pca_experiment.py`)

Held-out reconstruction MSE with $k=2$. *synthetic*: rank-2 latent data plus noise, **24 gross outliers injected into the training set only** (test rows are clean). *wine*: 75/25 split of `load_wine`. All data are standardised with training statistics.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA as ReferencePCA      # baseline only
from ml_reliability.statcore import PCA
from ml_reliability.statcore.pca.pca import VARIANTS

METHODS = ("reference",) + tuple(VARIANTS)

def standardize(train, test):
    mean = train.mean(0); scale = train.std(0); scale[scale <= 1e-12] = 1.0
    return (train - mean) / scale, (test - mean) / scale

def synthetic(seed):
    rng = np.random.default_rng(seed); latent = rng.normal(size=(400, 2))
    loading = np.array([[1.5, 0], [0, 1.2], [.8, .3], [0, .7], [.4, -.4], [.2, .5]])
    clean = latent @ loading.T + .12 * rng.normal(size=(400, 6)); order = rng.permutation(400)
    tr, te = order[:320], order[320:]; train = clean[tr].copy()
    idx = rng.choice(len(train), 24, replace=False); train[idx] += rng.normal(0, 8, (24, 6))
    xtr, xte = standardize(train, clean[te]); y = (latent[tr, 0] > 0).astype(int)
    return xtr, xte, y

def wine(seed):
    data = load_wine(); order = np.random.default_rng(seed).permutation(len(data.data)); split = int(.75 * len(order))
    tr, te = order[:split], order[split:]; xtr, xte = standardize(data.data[tr], data.data[te])
    return xtr, xte, data.target[tr]

def fit_method(method, xtr, y, seed):
    if method == "reference":
        return ReferencePCA(n_components=2, svd_solver="full").fit(xtr)
    return PCA(n_components=2, variant=method, random_state=seed).fit(xtr, y=y)

def score(method, xtr, xte, y, seed):
    model = fit_method(method, xtr, y, seed)
    recon = model.inverse_transform(model.transform(xte))
    return float(np.mean((xte - recon) ** 2))

rows = []
for ds, factory in (("synthetic", synthetic), ("wine", wine)):
    for method in METHODS:
        for seed in SEEDS:
            rows.append(dict(dataset=ds, method=method, seed=seed, mse=score(method, *factory(seed), seed)))
res = pd.DataFrame(rows)
tab = res.groupby(["dataset", "method"], sort=False)["mse"].agg(["mean", "std"])
tab["mean +/- sd"] = tab.apply(lambda r: f"{r['mean']:.4f} +/- {r['std']:.4f}", axis=1)
print(f"Held-out reconstruction MSE over seeds {SEEDS}")
tab[["mean +/- sd"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, ds in zip(axes, ("synthetic", "wine")):
    t = tab.loc[ds].loc[list(METHODS)]
    ax.bar(range(len(t)), t["mean"], yerr=t["std"], capsize=3, color=["gray"] + ["C0"] * (len(t) - 1))
    ax.set_xticks(range(len(t))); ax.set_xticklabels(t.index, rotation=35, ha="right")
    ax.set_title(f"{ds}: held-out reconstruction MSE (lower is better)"); ax.set_ylabel("MSE")
plt.tight_layout(); plt.show()

## 4. Look inside each variant

One fit per variant on the contaminated synthetic training set: explained variance, sparsity of the loadings, PPCA noise variance and how strongly the weighted variants down-weight rows.

In [ ]:
xtr, xte, y = synthetic(SEEDS[0])
info = []
models = {m: fit_method(m, xtr, y, SEEDS[0]) for m in METHODS}
for m, model in models.items():
    comps = model.components_
    row = dict(method=m, expl_var_ratio=np.round(model.explained_variance_ratio_, 3).tolist(),
               zero_loadings=int(np.sum(np.abs(comps) < 1e-12)),
               orthonormal_err=float(np.abs(comps @ comps.T - np.eye(2)).max()))
    row["noise_var"] = round(float(getattr(model, "noise_variance_", np.nan)), 4) if m == "probabilistic" else np.nan
    w = getattr(model, "row_weights_", None)
    row["min_row_weight"] = float(w.min()) if w is not None else np.nan
    row["rows_downweighted"] = int(np.sum(w < 0.999)) if w is not None else 0
    info.append(row)
pd.DataFrame(info).set_index("method")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, m in zip(axes, ("reference", "supervised_row_aware", "robust")):
    Z = models[m].transform(xtr)
    ax.scatter(Z[:, 0], Z[:, 1], c=y, s=10, cmap="coolwarm"); ax.set_title(f"{m}: training scores")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.tight_layout(); plt.show()

# per-sample reconstruction error as an outlier / reliability score (StatCore-only method)
err = models["robust"].score_samples(xtr)
print("robust PCA: mean per-sample error, top-5% rows vs rest ->",
      round(float(err[err >= np.quantile(err, .95)].mean()), 3), "vs", round(float(err[err < np.quantile(err, .95)].mean()), 3))

## 5. Summary and limitations

In [ ]:
best = tab.reset_index().sort_values("mean").groupby("dataset").first()[["method", "mean"]]
ref = tab.xs("reference", level="method")["mean"]
print("Best method per dataset (lowest held-out MSE):")
print(best.assign(reference_mse=ref, ratio_to_reference=best["mean"] / ref).round(4))

**What to read from the table.** On the contaminated synthetic set, estimators that limit the influence of the 24 training outliers (`robust`, `supervised_row_aware`) should reconstruct the *clean* test rows better than the reference; on clean real data (wine) the reference is essentially optimal and the variants are equal or worse.

**Honest limitations**
- Only 2 datasets, $k=2$, and a few seeds here (5 in the paper): differences within one standard deviation are not evidence of a real gap.
- The reference PCA is the optimal in-sample reconstruction; variants improve *robustness*, not clean-data error.
- `sparse` components are non-orthogonal (max deviation ≈ 1e-2 here) and its reconstruction is essentially the reference's (marginally worse on wine); the threshold `alpha` is not tuned and it is **not** robust to the injected outliers. `probabilistic` has the same subspace as the reference (so identical reconstruction MSE up to numerical error) and adds a noise model; it is not a better reconstructor.
- On clean wine, `robust` is the worst variant and `supervised_row_aware` is only ~0.5 % better than the reference — within one SD.
- `robust` stores the coordinate-wise median as the centring used by `transform`, while the components come from the weighted mean — a deliberate implementation detail, not a theorem.
- Outlier contamination in this study is synthetic and gross; subtle contamination, missing data and large $d$ were not studied.